# Load all single shot data

Dependencies: Python 3.9+, NumPy, h5py, Matplotlib and JupyterLab. If needed, run `%pip install numpy h5py matplotlib` in a notebook cell. Both notebooks are self-contained; no other scripts or inventory files are needed.

This notebook reads every selected single-shot file, one at a time. It keeps a compact summary. Set `KEEP_RAW=True` to retain all raw records as well; the complete collection can use several GB of memory. Set `PLOT=True` for the fidelity overview. Raw arrays are always available inside the loading loop.

In [ ]:
from pathlib import Path
import re
import numpy as np
import h5py
import matplotlib.pyplot as plt

NUMBER = re.compile(r'[-+]?(?:(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?|inf(?:inity)?|nan)', re.I)

def decode_value(value):
    """Decode bytes and complete serialized vectors; preserve other text verbatim.

    Raise on abbreviated arrays rather than silently treating '...' as full data.
    Nested arrays remain original text, accessible through raw_fields.
    """
    if isinstance(value, np.ndarray):
        if value.dtype.kind not in 'SUO':
            return value.copy()
        return [decode_value(x) for x in value.tolist()]
    if isinstance(value, (bytes, np.bytes_)):
        value = bytes(value).decode('utf-8', errors='strict')
    if isinstance(value, str):
        text = value.strip()
        if text == 'None':
            return None
        if text.startswith('[') and text.endswith(']'):
            if '...' in text:
                raise ValueError('Serialized array contains ellipsis; original samples are unavailable')
            if '[' in text[1:-1]:
                return value
            remainder = NUMBER.sub('', text[1:-1])
            if remainder.replace(',', '').strip():
                raise ValueError('Unrecognized content in serialized numeric vector')
            return np.fromstring(text[1:-1].replace(',', ' '), sep=' ', dtype=float)
        return value
    if isinstance(value, np.generic):
        return value.item()
    return value

def read_file(path):
    """Read all populated records in one HDF5 file, keeping field names unchanged."""
    records = []
    with h5py.File(path, 'r') as handle:
        for qubit, group in handle.items():
            if not isinstance(group, h5py.Group) or 'Dates' not in group:
                continue
            dates = np.atleast_1d(group['Dates'][()])
            for index, date in enumerate(dates):
                if not np.isfinite(float(date)):
                    continue
                fields = {}
                for key, dataset in group.items():
                    if not isinstance(dataset, h5py.Dataset):
                        continue
                    raw = dataset[()]
                    if isinstance(raw, np.ndarray) and raw.ndim and raw.shape[0] == len(dates):
                        raw = raw[index]
                    fields[key] = decode_value(raw)
                records.append(dict(source_file=str(path), qubit=qubit,
                                    dataset_index=index, fields=fields))
    if not records:
        raise ValueError(f'No populated records: {path}')
    return records

def plot_single_shot(record):
    """Plot every stored IQ sample and unrotated I histograms; return the figure.

    Ground/excited labels describe preparation. No reclassification, rotation,
    fidelity fitting, averaging or point subsampling is performed.
    """
    import matplotlib.pyplot as plt
    fields = record['fields']
    for state in ('g', 'e'):
        for axis in ('I', 'Q'):
            value = fields.get(f'{axis}_{state}')
            if not isinstance(value, np.ndarray) or value.ndim != 1 or not value.size:
                raise ValueError('Plotting requires single-shot I_g, Q_g, I_e and Q_e vectors.')
        if fields[f'I_{state}'].size != fields[f'Q_{state}'].size:
            raise ValueError(f'{state} I/Q lengths differ')
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    for state, label, color in [('g', 'Prepared ground', '#2463a7'),
                                ('e', 'Prepared excited', '#ce5935')]:
        i, q = fields[f'I_{state}'], fields[f'Q_{state}']
        axes[0].scatter(i, q, s=3, alpha=.2, color=color, rasterized=True,
                        label=f'{label} (n={len(i):,})')
        axes[1].hist(i[np.isfinite(i)], bins=80, histtype='step', linewidth=1.6,
                     color=color, label=label)
    axes[0].set(xlabel='I (stored units)', ylabel='Q (stored units)', title='Raw single-shot IQ')
    axes[0].legend(markerscale=3, fontsize=8)
    axes[1].set(xlabel='I (stored units, unrotated)', ylabel='Shots per bin', title='I distributions')
    axes[1].legend(fontsize=8)
    for ax in axes:
        ax.grid(alpha=.15)
    fidelity = fields.get('Fidelity')
    suffix = f' | stored SSF contrast {fidelity:.4f}' if isinstance(fidelity, (float, int)) else ''
    fig.suptitle(f"{record['qubit']} | record {record['dataset_index']}{suffix}")
    return fig

In [ ]:
# Set PACKAGE_ROOT if Jupyter starts outside the extracted package.
PACKAGE_ROOT = None
if PACKAGE_ROOT is None:
    PACKAGE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / 'datasets').is_dir() and (p / 'documentation').is_dir()), None)
    if PACKAGE_ROOT is None:
        raise FileNotFoundError('Set PACKAGE_ROOT to your extracted package folder.')
PACKAGE_ROOT = Path(PACKAGE_ROOT).expanduser().resolve()


In [ ]:
RUN = None  # e.g. 'ssf_24h_q2_2026-11-11_12-52-52'; None selects all runs
QUBIT = None  # e.g. 'Q2'; None selects all populated qubits
MAX_FILES = None  # optionally use 10 for a quick preview
KEEP_RAW = False
PLOT = True
SAVE_PLOT = False

files = sorted((PACKAGE_ROOT / 'datasets').glob('*/qubit*round*/single_shot/data/*.h5'))
# All original acquisitions are included without duplicate exclusions.
files = [p for p in files if RUN is None or p.relative_to(PACKAGE_ROOT / 'datasets').parts[0] == RUN]
if MAX_FILES is not None:
    files = files[:MAX_FILES]
print('Files selected:', len(files))

In [ ]:
summary = []
raw_records = []
for file_index, path in enumerate(files):
    for record in read_file(path):
        if QUBIT is not None and record['qubit'] != QUBIT:
            continue
        f = record['fields']
        I_g, Q_g, I_e, Q_e = (f[k] for k in ['I_g', 'Q_g', 'I_e', 'Q_e'])
        if len(I_g) != len(Q_g) or len(I_e) != len(Q_e):
            raise ValueError(f'I/Q length mismatch: {path}')
        # Add your analysis of the complete arrays here.
        summary.append(dict(run=path.relative_to(PACKAGE_ROOT / 'datasets').parts[0],
                            qubit=record['qubit'], date_unix=float(f['Dates']),
                            fidelity=float(f['Fidelity']), ground_shots=len(I_g),
                            excited_shots=len(I_e), source_file=str(path.relative_to(PACKAGE_ROOT))))
        if KEEP_RAW:
            raw_records.append(record)
    if (file_index + 1) % 500 == 0:
        print('Files read:', file_index + 1)
if not summary:
    raise ValueError('No records match the chosen run/qubit filters.')
print('Loaded records:', len(summary))
print('Ground plus excited shots:', sum(r['ground_shots'] + r['excited_shots'] for r in summary))
print('Raw records retained:', len(raw_records))

## Optional overview plot

Saved epochs are displayed in UTC. This does not establish the timezone used in the original filenames. The plotted fidelity is the stored acquisition value.

Nov 11–12, 2026; filenames use America/New_York local time. The saved Fidelity field is adaptive histogram CDF contrast, not assignment accuracy. Nov 4 calibrations were reused without retuning.

In [ ]:
if PLOT or SAVE_PLOT:
    import datetime as dt
    from collections import defaultdict
    groups = defaultdict(list)
    for row in summary:
        groups[(row['run'], row['qubit'])].append(row)
    fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
    for (run, qubit), rows in sorted(groups.items()):
        dates = [dt.datetime.fromtimestamp(r['date_unix'], dt.timezone.utc) for r in rows]
        ax.scatter(dates, [r['fidelity'] for r in rows], s=5, label=f'{run.replace("QZE_IBM_ssf_tracking", "run")} {qubit}')
    ax.set(xlabel='Saved acquisition epoch converted to UTC', ylabel='Stored SSF contrast', ylim=(0, 1))
    ax.legend(fontsize=7, ncol=3)
    ax.grid(alpha=.2)
    fig.autofmt_xdate()
    if SAVE_PLOT:
        fig.savefig(PACKAGE_ROOT / 'single_shot_overview.png', dpi=160)
    if PLOT:
        plt.show()
    else:
        plt.close(fig)